# Pattern 7: Gateway + Lambda Interceptor

Add a Lambda function as a request/response interceptor on the Gateway. The interceptor
runs before (REQUEST) and/or after (RESPONSE) each tool invocation, enabling custom logic
like query guardrails, dynamic filter injection, audit logging, rate limiting, or result
redaction.

**What you get:** Custom pre/post processing on every Gateway request — logic the gateway
itself doesn't provide.

**This notebook demonstrates a REQUEST-side query-content guardrail:** the interceptor
inspects the retrieval query and refuses it (before the KB is ever touched) if it mentions
a prohibited term. A benign query passes straight through.

## Prerequisites

- Run [Pattern 1](01-direct-sdk.ipynb) first — it creates the shared bucket, uploads the
  sample documents, and creates the KB execution role. (The setup cell here re-runs it
  idempotently, and additionally creates the **Gateway role**.)
- IAM permissions for Bedrock, AgentCore (`bedrock-agentcore-control`), Lambda, S3, and IAM.

## Architecture

```
Agent ──► Gateway ──► Lambda (REQUEST) ──► KB Target ──► Managed KB
                         │
                         ├── query OK        → forward to KB (transformedGatewayRequest)
                         └── query blocked   → short-circuit (transformedGatewayResponse),
                                               KB is never queried
```

In [ ]:
import boto3
import time
import json
import zipfile
import io
import util   # util.py in this folder — shared bucket + upload + roles

# --- Configuration ---
REGION = "us-west-2"
S3_BUCKET = "<existing-or-unique-name-for-your-kb-bucket->"
S3_PREFIX = "documents/"

session = boto3.Session()

# Reuse the SAME bucket + docs + KB execution role as Pattern 1 (idempotent),
# then create the Gateway role the AgentCore Gateway assumes to retrieve.
info = util.setup(
    bucket_name=S3_BUCKET,
    prefix=S3_PREFIX,
    metadata=util.SAMPLE_FILE_METADATA,
    region_name=REGION,
)
ROLE_ARN     = info["role_arn"]
S3_BUCKET    = info["bucket"]
S3_PREFIX    = info["prefix"]
GW_ROLE_ARN  = util.create_gateway_role(region_name=REGION)
GW_ROLE_NAME = GW_ROLE_ARN.split("/")[-1]

# Clients
cp = session.client("bedrock-agent", region_name=REGION)
dp = session.client("bedrock-agent-runtime", region_name=REGION)
ac = session.client("bedrock-agentcore-control", region_name=REGION)
lambda_client = session.client("lambda", region_name=REGION)
iam = session.client("iam")
ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]
S3_ACCOUNT = ACCOUNT_ID

print(f"boto3 {boto3.__version__}")
print(f"KB role:      {ROLE_ARN}")
print(f"Gateway role: {GW_ROLE_ARN}")


In [ ]:
# Step 1: Create KB + Data Source + Ingest (same as Pattern 1)
response = cp.create_knowledge_base(
    name=f"p7-interceptor-{int(time.time())}",
    roleArn=ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "MANAGED",
        "managedKnowledgeBaseConfiguration": {}   # empty = managed default embedding
    }
)
kb_id = response["knowledgeBase"]["knowledgeBaseId"]
print(f"KB: {kb_id}")

for _ in range(30):
    if cp.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"] == "ACTIVE":
        break
    time.sleep(5)
print("KB ACTIVE")

response = cp.create_data_source(
    knowledgeBaseId=kb_id,
    name="s3-source",
    dataSourceConfiguration={
        "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
        "managedKnowledgeBaseConnectorConfiguration": {
            "connectorParameters": {
                "type": "S3",
                "version": "1",
                "connectionConfiguration": {
                    "bucketName": S3_BUCKET,
                    "bucketOwnerAccountId": S3_ACCOUNT
                },
                "filterConfiguration": {"inclusionPrefixes": [S3_PREFIX]},
                "deletionProtectionConfiguration": {"enableDeletionProtection": False}
            },
            "deletionProtectionConfiguration": {"deletionProtectionStatus": "DISABLED"}
        }
    },
    vectorIngestionConfiguration={
        "parsingConfiguration": {"parsingStrategy": "SMART_PARSING"}
    }
)
ds_id = response["dataSource"]["dataSourceId"]
print(f"DS: {ds_id}")

for _ in range(12):
    if cp.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]["status"] == "AVAILABLE":
        break
    time.sleep(5)
print("DS AVAILABLE")

response = cp.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
job_id = response["ingestionJob"]["ingestionJobId"]
for _ in range(40):
    job = cp.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
    )["ingestionJob"]
    if job["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)
print(f"Ingestion: {job['status']}")

In [ ]:
# Step 2: Create a Lambda execution role (basic CloudWatch Logs access)
lambda_role_name = "AmazonBedrockInterceptorLambdaRole"
trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
try:
    r = iam.create_role(RoleName=lambda_role_name, AssumeRolePolicyDocument=json.dumps(trust))
    iam.attach_role_policy(
        RoleName=lambda_role_name,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    LAMBDA_ROLE_ARN = r["Role"]["Arn"]
    print(f"Created Lambda role: {lambda_role_name}")
    print("  Waiting 12s for IAM propagation...")
    time.sleep(12)
except iam.exceptions.EntityAlreadyExistsException:
    LAMBDA_ROLE_ARN = iam.get_role(RoleName=lambda_role_name)["Role"]["Arn"]
    print(f"Lambda role already exists: {lambda_role_name}")

print(f"LAMBDA_ROLE_ARN = {LAMBDA_ROLE_ARN}")

In [ ]:
# Step 3: Create the interceptor Lambda — a REQUEST-side query-content guardrail.
#
# The interceptor runs BEFORE the gateway calls the KB target. It reads the MCP
# request body, and for a tools/call it inspects the retrieval query text:
#   - contains a blocked term  → short-circuit with a JSON-RPC error (KB never queried)
#   - otherwise                → pass the request through unchanged
#
# Contract (see the "Interceptor Event Format" section below):
#   input:  event["mcp"]["gatewayRequest"]["body"] holds the JSON-RPC request
#   output: return {"interceptorOutputVersion": "1.0", "mcp": {...}} where
#           - "transformedGatewayRequest": {"body": ...} forwards a (modified) request
#           - "transformedGatewayResponse": {...} short-circuits and replies immediately
handler_code = r'''
import json, logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Queries mentioning these terms are refused before they reach the KB.
BLOCKED_TERMS = ["salary", "ssn", "compensation"]

def handler(event, context):
    logger.info("Interceptor event: " + json.dumps(event, default=str))

    body = event.get("mcp", {}).get("gatewayRequest", {}).get("body", {}) or {}
    req_id = body.get("id", 1)

    # Pass everything that is not a tool call straight through (initialize,
    # tools/list, etc.) by echoing the original request body.
    passthrough = {
        "interceptorOutputVersion": "1.0",
        "mcp": {"transformedGatewayRequest": {"body": body}},
    }
    if body.get("method") != "tools/call":
        return passthrough

    args = (body.get("params", {}) or {}).get("arguments", {}) or {}
    query = ((args.get("retrievalQuery", {}) or {}).get("text", "") or "").lower()
    hit = next((term for term in BLOCKED_TERMS if term in query), None)
    if not hit:
        return passthrough

    # Short-circuit: reply immediately with a JSON-RPC error; the KB is never hit.
    logger.info(f"BLOCKED query (matched '{hit}'): {query!r}")
    return {
        "interceptorOutputVersion": "1.0",
        "mcp": {
            "transformedGatewayResponse": {
                "statusCode": 200,
                "body": {
                    "jsonrpc": "2.0",
                    "id": req_id,
                    "error": {
                        "code": -32001,
                        "message": f"Blocked by interceptor policy: query contains prohibited term '{hit}'",
                    },
                },
            }
        },
    }
'''

# Package as zip
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", handler_code)
zip_buffer.seek(0)

func_name = f"p7-interceptor-{int(time.time())}"
fn = lambda_client.create_function(
    FunctionName=func_name,
    Runtime="python3.12",
    Role=LAMBDA_ROLE_ARN,
    Handler="lambda_function.handler",
    Code={"ZipFile": zip_buffer.read()},
    Timeout=30
)
LAMBDA_ARN = fn["FunctionArn"]
print(f"Lambda: {LAMBDA_ARN}")

# Wait for Active
for _ in range(12):
    state = lambda_client.get_function(FunctionName=func_name)["Configuration"]["State"]
    if state == "Active":
        break
    time.sleep(5)
print(f"Lambda state: {state}")

In [ ]:
# Step 4: Grant the GATEWAY SERVICE ROLE permission to invoke the interceptor.
# This is the permission that actually matters: the gateway invokes the
# interceptor Lambda using its own service role, so that role needs
# lambda:InvokeFunction on the function. (Without this, tool calls fail with a
# 500 before the interceptor ever runs.)
iam.put_role_policy(
    RoleName=GW_ROLE_NAME,
    PolicyName="InterceptorInvoke",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["lambda:InvokeFunction"],
            "Resource": [LAMBDA_ARN],
        }],
    }),
)
print(f"Granted {GW_ROLE_NAME} lambda:InvokeFunction on the interceptor")
print("  Waiting 12s for IAM propagation...")
time.sleep(12)

In [ ]:
# Step 5: Create Gateway with the REQUEST interceptor (AWS_IAM auth, like Pattern 3)
gw_response = ac.create_gateway(
    name=f"p7-interceptor-gw-{int(time.time())}",
    roleArn=GW_ROLE_ARN,
    protocolType="MCP",
    authorizerType="AWS_IAM",
    interceptorConfigurations=[
        {
            "interceptor": {"lambda": {"arn": LAMBDA_ARN}},
            "interceptionPoints": ["REQUEST"],   # guardrail runs before the KB call
            "inputConfiguration": {"passRequestHeaders": True}
        }
    ]
)

gw_id = gw_response["gatewayId"]
print(f"Gateway: {gw_id}")

gw_url = None
for _ in range(24):
    gw = ac.get_gateway(gatewayIdentifier=gw_id)
    if gw["status"] == "READY":
        gw_url = gw.get("gatewayUrl", "N/A")
        break
    elif "FAIL" in gw["status"]:
        print(f"FAILED: {gw.get('statusReasons', [])}")
        break
    time.sleep(5)
print(f"Status: {gw['status']}")
print(f"Gateway URL: {gw_url}")

In [ ]:
# Step 6: Create KB Target on the interceptor gateway
target_response = ac.create_gateway_target(
    gatewayIdentifier=gw_id,
    name="kb-retrieve",
    targetConfiguration={
        "mcp": {
            "connector": {
                "source": {"connectorId": "bedrock-knowledge-bases"},
                "configurations": [{
                    "name": "Retrieve",
                    # Tool description exposed to the agent over MCP — this is what
                    # the LLM reads to decide when to call this KB.
                    "description": (
                        "Search two corporate documents: (1) Octank Financial's 10-K annual "
                        "report — financial statements, asset/liability schedules, exhibits, and "
                        "investor disclosures; and (2) a U.S. tornado background & forecasting "
                        "report — where tornadoes form, annual frequency (~1,200/yr), and NOAA data."
                    ),
                    "parameterValues": {
                        "knowledgeBaseId": kb_id,
                        "retrievalConfiguration": {
                            "managedSearchConfiguration": {
                                "numberOfResults": 5
                            }
                        }
                    }
                }]
            }
        }
    },
    credentialProviderConfigurations=[
        {"credentialProviderType": "GATEWAY_IAM_ROLE"}
    ]
)

target_id = target_response["targetId"]
print(f"Target: {target_id}")

for _ in range(12):
    t = ac.get_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
    if t["status"] == "READY":
        break
    time.sleep(5)
print(f"Target status: {t['status']}")

In [ ]:
# Step 7: A benign query PASSES the interceptor and reaches the KB.
# We call through the gateway with the official MCP client. An AWS_IAM gateway
# requires SigV4-signed requests, so we wrap botocore's signer in an httpx.Auth
# (same helper as Pattern 3). The REQUEST interceptor sees this query, finds no
# blocked term, and passes it through to the KB.
import httpx
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from mcp.shared.exceptions import McpError


class SigV4HTTPXAuth(httpx.Auth):
    """httpx auth handler that SigV4-signs each request for the bedrock-agentcore service."""
    requires_request_body = True

    def __init__(self, credentials, service, region):
        self._credentials, self._service, self._region = credentials, service, region

    def auth_flow(self, request):
        aws_req = AWSRequest(
            method=request.method, url=str(request.url),
            data=request.content, headers=dict(request.headers),
        )
        SigV4Auth(self._credentials, self._service, self._region).add_auth(aws_req)
        request.headers.update(dict(aws_req.headers))
        yield request


sigv4 = SigV4HTTPXAuth(session.get_credentials(), "bedrock-agentcore", REGION)


async def gateway_retrieve(query_text):
    """Call the KB Retrieve tool through the gateway.

    Returns the CallToolResult on success, or the McpError itself if the
    interceptor blocked the request. We catch McpError INSIDE the session — if it
    escaped this context it would be wrapped in nested ExceptionGroups by the
    client's task group, which is awkward to unpack at the call site.
    """
    async with httpx.AsyncClient(auth=sigv4) as http_client:
        async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
            async with ClientSession(read, write) as session_mcp:
                await session_mcp.initialize()
                listing = await session_mcp.list_tools()
                tool_name = next(t.name for t in listing.tools if t.name.split("___")[-1] == "Retrieve")
                try:
                    return await session_mcp.call_tool(
                        name=tool_name,
                        arguments={"retrievalQuery": {"text": query_text}},
                    )
                except McpError as e:
                    return e


result = await gateway_retrieve("What are Octank's key financial results?")
print(f"isError: {result.isError}")
print("=== ALLOWED query — passed the interceptor, retrieved from KB ===")
print(result.content[0].text[:800] if result.content else result)

In [ ]:
# Step 8: A query containing a blocked term is REFUSED by the interceptor.
# "salary" is in BLOCKED_TERMS, so the REQUEST interceptor short-circuits with a
# JSON-RPC error BEFORE the gateway ever calls the KB. gateway_retrieve() returns
# that McpError (code -32001) instead of KB results.
result = await gateway_retrieve("What is the CEO salary and compensation?")

if isinstance(result, McpError) and result.error.code == -32001:
    print("BLOCKED by the interceptor (KB was never queried)")
    print(f"   {result.error.message}")
elif isinstance(result, McpError):
    print(f"Unexpected MCP error [{result.error.code}]: {result.error.message}")
else:
    print("Unexpected — query was allowed:")
    print(result.content[0].text[:400] if result.content else result)

## Interceptor Use Cases

This notebook implements the **query guardrail** row (REQUEST, short-circuit). The same
interceptor mechanism supports many other patterns:

| Use Case | Interception Point | What the Lambda Does |
|---|---|---|
| **Query guardrail** (this notebook) | `REQUEST` | Inspect the query; short-circuit with an error if it violates policy |
| **Dynamic filter injection** | `REQUEST` | Read user identity from headers, inject metadata filters based on department/role |
| **Query rewriting** | `REQUEST` | Modify the query text, add context, or translate — return `transformedGatewayRequest` |
| **Audit logging** | `REQUEST` + `RESPONSE` | Log who queried what, when, and what was returned |
| **Rate limiting** | `REQUEST` | Check a DynamoDB counter, short-circuit if over limit |
| **Result redaction** | `RESPONSE` | Strip PII / redact sensitive fields before returning to the caller |
| **Response enrichment** | `RESPONSE` | Add metadata, citations, or confidence scores |

The `passRequestHeaders: true` setting forwards HTTP headers to the Lambda, which is what
makes identity-based logic (reading JWT claims from the `Authorization` header) possible —
that's the basis for the filter-injection variant referenced by Patterns 5 and 6.

In [ ]:
# Cleanup — order: target → gateway → Lambda → DS → KB
ac.delete_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
time.sleep(3)
ac.delete_gateway(gatewayIdentifier=gw_id)
time.sleep(3)
lambda_client.delete_function(FunctionName=LAMBDA_ARN)
cp.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
cp.delete_knowledge_base(knowledgeBaseId=kb_id)
print(f"Deleted: Gateway {gw_id}, Lambda {LAMBDA_ARN}, KB {kb_id}")

# We intentionally DO NOT delete two reusable artifacts, so re-runs stay
# idempotent and a concurrent run can't strip the permission from a live gateway:
#   - the InterceptorInvoke inline policy on GW_ROLE_NAME (Step 4) — a re-run just
#     overwrites it to point at the new Lambda ARN, which is harmless
#   - the Lambda execution role AmazonBedrockInterceptorLambdaRole (Step 2)
# To tear the whole pattern down for good, delete them manually:
#   iam.delete_role_policy(RoleName=GW_ROLE_NAME, PolicyName="InterceptorInvoke")
#   # (detach AWSLambdaBasicExecutionRole, then) iam.delete_role(RoleName="AmazonBedrockInterceptorLambdaRole")

## Interceptor Event Format (MCP targets)

The interceptor Lambda receives the MCP request (and, for RESPONSE interceptors, the
response) nested under an `mcp` key. **REQUEST** input:

```json
{
  "interceptorInputVersion": "1.0",
  "mcp": {
    "rawGatewayRequest": { "body": "<raw_request_body>" },
    "gatewayRequest": {
      "path": "/mcp",
      "httpMethod": "POST",
      "headers": { "Authorization": "<bearer_token>", "...": "..." },
      "body": { "jsonrpc": "2.0", "id": 1, "method": "tools/call",
                "params": { "name": "<tool>", "arguments": { "retrievalQuery": { "text": "..." } } } }
    }
  }
}
```

- `headers` is present only when `passRequestHeaders: true`.
- A **RESPONSE** interceptor additionally receives `mcp.gatewayResponse` (the KB result).

### What the Lambda returns

The output is nested under `mcp` too, with `interceptorOutputVersion: "1.0"`:

```json
{
  "interceptorOutputVersion": "1.0",
  "mcp": {
    "transformedGatewayRequest":  { "body": { "...modified JSON-RPC request..." } },
    "transformedGatewayResponse": { "statusCode": 200, "body": { "...JSON-RPC response..." } }
  }
}
```

- Return **`transformedGatewayRequest`** to forward a (possibly modified) request to the KB
  — this is the pass-through / filter-injection / query-rewrite path.
- Return **`transformedGatewayResponse`** to **short-circuit**: the gateway replies with that
  content immediately and never calls the KB — this is the guardrail / block path used in
  Step 8 (a JSON-RPC `error` with code `-32001`).
- If both are present, `transformedGatewayResponse` wins.

### Required permission

The **gateway service role** (not the caller) invokes the interceptor, so it must hold
`lambda:InvokeFunction` on the function (granted in Step 4). Missing this permission surfaces
as an HTTP 500 on the tool call — before the Lambda ever runs.

> HTTP targets (AgentCore Runtime, passthrough) use a different `http` payload shape where
> bodies are base64-encoded strings. This notebook uses an MCP target, so the payload is the
> `mcp` shape shown above.